<a href="https://colab.research.google.com/github/EDWINTUZYEH/tecnm-lenguajes-automatas-2/blob/main/laboratorios/sesion01/sesion01_laboratorio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lenguajes y Autómatas II (SCD-1016)
## Sesión 01: Encuadre, Diagnóstico de Punteros y Metodología IA
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Inversión de buffer in-place (char*)*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Punteros, O(n) tiempo, O(1) memoria* y formular prompt.
3. **Fase 3 (El Auditor):** Someter la solución al Test Harness: *Valores NULL, buffers vacios y longitudes impares*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion01.md`.

### Paso 0: Inicialización del Entorno (C y Rust)

In [66]:
import os
if os.system('which rustc > /dev/null 2>&1') != 0:
    print('⏳ Instalando Rust en Colab...')
    !apt-get update -qq && apt-get install -y -qq rustc > /dev/null 2>&1
print('✅ Entorno listo:')
!gcc --version | head -n 1
!rustc --version

✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
rustc 1.75.0 (82e1608df 2023-12-21) (built from a source tarball)


---
## SECCIÓN A: TRACK C
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Inversión de buffer in-place (char*)

In [67]:
%%writefile artesano_sesion01_c.c
#include <stdio.h>
#include <stdlib.h>

void reverse_inplace(char *s) {
    if (s == NULL || *s == '\0') return;

    int n = 0;
    while (s[n] != '\0') n++;

    char *i = s;
    char *j = s + n - 1;
    while (i < j) {
        char t = *i;
        *i = *j;
        *j = t;
        i++;
        j--;
    }
}

int main() {
    printf("[SESION 01] Reto Artesano: Inversión de buffer in-place (char*)\n");
    char a[] = "abcde";
    reverse_inplace(a);
    printf("%s\n", a);
    return 0;
}


Overwriting artesano_sesion01_c.c


In [68]:
!gcc -Wall -Wextra artesano_sesion01_c.c -o bin_c && ./bin_c

[SESION 01] Reto Artesano: Inversión de buffer in-place (char*)
edcba


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [69]:
%%writefile ia_sesion01_c.c
#include <stddef.h>
#include <string.h>

/* Invierte in-place una cadena terminada en '\0'.
 * Retorna 0 si OK, -1 si s es NULL.
 * Tiempo O(n), memoria extra O(1) (un char temporal y dos punteros). */
int reverse_inplace(char *s) {
    if (s == NULL) return -1;
    size_t n = strlen(s);
    if (n < 2) return 0;
    char *i = s, *j = s + n - 1;
    while (i < j) {
        char t = *i;
        *i++ = *j;
        *j-- = t;
    }
    return 0;
}

/* Variante con longitud explícita (buffers sin '\0'). */
int reverse_n_inplace(char *buf, size_t len) {
    if (buf == NULL) return -1;
    if (len < 2) return 0;
    char *i = buf, *j = buf + len - 1;
    while (i < j) {
        char t = *i;
        *i++ = *j;
        *j-- = t;
    }
    return 0;
}

Overwriting ia_sesion01_c.c


### Fase 3: El Auditor (Test Harness en C)
Pruebas de estrés: Valores NULL, buffers vacios y longitudes impares

In [70]:
%%writefile harness_sesion01_c.c
#include "stdio.h"
#include "string.h"
#include "ia_sesion01_c.c"

static int prueba(const char *nombre, const char *entrada, const char *esperado) {
    char buf[32];
    strcpy(buf, entrada);
    reverse_inplace(buf);
    int ok = strcmp(buf, esperado) == 0;
    printf("%s: %s\n", nombre, ok ? "PASO" : "FALLO");
    return ok;
}

int main() {
    printf("=== AUDITORIA CRITICA: SESION 01 ===\n");
    int ok = 1;

    reverse_inplace(NULL);
    printf("NULL: PASO\n");

    ok &= prueba("Buffer vacio", "", "");
    ok &= prueba("Un caracter", "a", "a");
    ok &= prueba("Longitud par", "abcd", "dcba");
    ok &= prueba("Longitud impar", "abcde", "edcba");

    printf(ok ? "RESULTADO: TODO PASO\n" : "RESULTADO: HAY FALLOS\n");
    return 0;
}

Overwriting harness_sesion01_c.c


In [71]:
!gcc -Wall -Wextra harness_sesion01_c.c -o test_bin && ./test_bin

=== AUDITORIA CRITICA: SESION 01 ===
NULL: PASO
Buffer vacio: PASO
Un caracter: PASO
Longitud par: PASO
Longitud impar: PASO
RESULTADO: TODO PASO
